In [3]:
##preprocessing data
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [9]:
## load data and make it lowercase
with open('pizza.txt','r') as file:
  text=file.read().lower()

In [10]:
text

'pizza, the delectable and iconic dish that has transcended borders and captivated taste buds worldwide, is a testament to the extraordinary fusion of flavors, creativity, and cultural significance. originating from the sun-kissed lands of italy, pizza has evolved into an art form that unites people from diverse backgrounds in a shared love for its mouthwatering combinations. its history stretches back centuries, with roots tracing back to ancient civilizations like the greeks, romans, and egyptians, who all had their versions of flatbreads adorned with various ingredients. however, it was the vibrant city of naples, italy, that birthed the pizza we know and adore today.\n\nwith its soft and chewy neapolitan crust, topped with the perfect balance of tomatoes, mozzarella cheese, and fresh basil, the margherita pizza pays homage to queen margherita of italy and embodies the colors of the italian flag. as pizza migrated from the shores of naples, it found its way to the united states with

In [11]:
##tokenize text
tokenize=Tokenizer()
tokenize.fit_on_texts([text])
total_words=len(tokenize.word_index)+1
total_words


687

In [ ]:
tokenize.word_index

In [17]:
##create input sequence order
input_sequences=[]
for line in text.split('\n'):
  token_list=tokenize.texts_to_sequences([line])[0]  ##tokenize in sequence [1,2,3,4,5,6,7,]
  for i in range(1,len(token_list)):
    n_gram_seq=token_list[:i+1]  ##next word prediction 
    input_sequences.append(n_gram_seq)


In [19]:
##padding 
max_len_seq=max([len(x) for x in input_sequences])
max_len_seq
input_sequences=np.array(pad_sequences(input_sequences, maxlen=max_len_seq,padding='pre'))

In [20]:
input_sequences

array([[  0,   0,   0, ...,   0,   3,   1],
       [  0,   0,   0, ...,   3,   1, 233],
       [  0,   0,   0, ...,   1, 233,   2],
       ...,
       [  0,   0,   0, ..., 685,   4,  19],
       [  0,   0,   0, ...,   4,  19,  72],
       [  0,   0,   0, ...,  19,  72, 686]],
      shape=(1686, 146), dtype=int32)

In [26]:
##create predictor or label
X,Y=input_sequences[:,:-1],input_sequences[:,-1]

In [27]:
##converts target labels/word (y) into one-hot encoded vectors
import tensorflow as tf
Y=tf.keras.utils.to_categorical(Y,num_classes=total_words)

In [25]:
Y

array([[[1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [1., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.]],

       [[1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 1., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]],

       ...,

       [[1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 1., 0., 0.],
        [0., 0., 0., ..., 0., 1., 0.],
        [0., 0., 0., ..., 0., 0.

In [28]:
##Split data for train and test
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=.20)

In [29]:
##DEFINE YEARLYSTOPPING
from tensorflow.keras.callbacks import EarlyStopping
early_stop=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [39]:
##Train LSTM RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Embedding,GRU,LSTM,Dense,Dropout

model=Sequential()
model.add(Input(shape=(max_len_seq - 1,)))
model.add(Embedding(total_words,50))
model.add(LSTM(100,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(50))
model.add(Dense(total_words,activation='softmax'))


In [40]:
##compie the model
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 145, 50)        │        34,350 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 145, 100)       │        60,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 145, 100)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 50)             │        30,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 687)            │        35,037 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 159,987 (624.95 KB)

 Trainable params: 159,987 (624.95 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
##Trin the model
history=model.fit(X_train,Y_train,epochs=25,validation_data=(X_test,Y_test),verbose=1,callbacks=[early_stop])

Epoch 1/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 25s 223ms/step - accuracy: 0.0519 - loss: 6.3152 - val_accuracy: 0.0621 - val_loss: 6.0325
Epoch 2/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 172ms/step - accuracy: 0.0519 - loss: 5.7495 - val_accuracy: 0.0473 - val_loss: 6.2259
Epoch 3/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 167ms/step - accuracy: 0.0571 - loss: 5.6448 - val_accuracy: 0.0473 - val_loss: 6.3735
Epoch 4/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 8s 180ms/step - accuracy: 0.0542 - loss: 5.6270 - val_accuracy: 0.0473 - val_loss: 6.4698
Epoch 5/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 9s 192ms/step - accuracy: 0.0571 - loss: 5.6151 - val_accuracy: 0.0473 - val_loss: 6.5577
Epoch 6/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 9s 202ms/step - accuracy: 0.0534 - loss: 5.6117 - val_accuracy: 0.0473 - val_loss: 6.6238
Epoch 7/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 7s 159ms/step - accuracy: 0.0571 - loss: 5.6050 - val_accuracy: 0.0473 - val_loss: 6.6550
Epoch 8/25
43/43 ━━━━━━━━━━━━━━━━━━━━ 7s 165ms/step - accuracy: 0.0504 - loss: 5.5898 - val_accuracy: 0

In [60]:
##GRU ANN
model=Sequential()
model.add(Input(shape=(max_len_seq-1,)))
model.add(Embedding(total_words,100))
model.add(GRU(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(100))
model.add(Dense(total_words,activation='softmax'))

In [62]:
##compile GRU ANN
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 145, 100)       │        68,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 145, 150)       │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 145, 150)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 100)            │        75,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 687)            │        69,387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 327,087 (1.25 MB)

 Trainable params: 327,087 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
##train GRU ANN
history=model.fit(X_train,Y_train,epochs=50,validation_data=(X_test,Y_test),verbose=1,callbacks=[early_stop])

Epoch 1/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 25s 428ms/step - accuracy: 0.0386 - loss: 6.2518 - val_accuracy: 0.0621 - val_loss: 6.0815
Epoch 2/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 16s 364ms/step - accuracy: 0.0564 - loss: 5.7742 - val_accuracy: 0.0473 - val_loss: 6.3423
Epoch 3/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 13s 296ms/step - accuracy: 0.0564 - loss: 5.6754 - val_accuracy: 0.0533 - val_loss: 6.5082
Epoch 4/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 13s 292ms/step - accuracy: 0.0564 - loss: 5.6152 - val_accuracy: 0.0562 - val_loss: 6.5930
Epoch 5/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 15s 355ms/step - accuracy: 0.0593 - loss: 5.5619 - val_accuracy: 0.0651 - val_loss: 6.6080
Epoch 6/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 23s 546ms/step - accuracy: 0.0920 - loss: 5.4539 - val_accuracy: 0.0828 - val_loss: 6.7578
Epoch 7/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 18s 425ms/step - accuracy: 0.1313 - loss: 5.2399 - val_accuracy: 0.1006 - val_loss: 6.6504
Epoch 8/50
43/43 ━━━━━━━━━━━━━━━━━━━━ 19s 442ms/step - accuracy: 0.1543 - loss: 4.9542 - val_accu

In [64]:
##function to predict nextword
def predict_next_word(model,tokenize,text,max_len_seq):
  token_list=tokenize.texts_to_sequences([text])[0]
  if len(token_list)>=max_len_seq:
    token_list=token_list[-(max_len_seq-1):] ## in sequence of (last max_len_seq-1) words 
  token_list=pad_sequences([token_list],maxlen=max_len_seq-1,padding='pre')   ##exact lenght as used in training model
  predicted=model.predict(token_list,verbose=0)       ##predict in probability
  predicted_word_index=np.argmax(predicted,axis=1)  ##predict index of max probability

  for word,index in tokenize.word_index.items():
    if index==predicted_word_index:
      return word

    return None



In [65]:
input_text='With its soft and chewy Neapolitan'
print(f'input text: {input_text}')
max_length=model.input_shape[1]+1
next_predicted_word=predict_next_word(model,tokenize,input_text,max_length)
print(f'predicted word :{next_predicted_word}')

input text: With its soft and chewy Neapolitan
predicted word :the


In [67]:
#save the model
model.save('next_word_lstm.h5')

##save the tiokenizer
import pickle
with open('tokenizer.pickle','wb')as file:
  pickle.dump(tokenize,file,protocol=pickle.HIGHEST_PROTOCOL)